In [3]:
# Load the python package
import os
import dynetan
from dynetan.toolkit import *
from dynetan.viz import *
from dynetan.proctraj import *
from dynetan.gencor import *
from dynetan.contact import *

# Create the object that processes MD trajectories.
dnap = DNAproc()
dnap2 = DNAproc()

import sys, os
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
from scipy.interpolate import griddata
import seaborn as sb
from statistics import mean, stdev
from matplotlib import colors
from IPython.display import set_matplotlib_formats

In [4]:
# Path where input files will searched and results be written.
source_dir="/home/usuario/ns3/qmmm/paper/md/"

# List of directories
all_dir = ["atp/", "rna7A/", "rna16A/clustering2/5endDown/", "rna16A/clustering2/5endUp/"]

i=1
for directory in all_dir:
    workDir = source_dir + directory + "analisis/"
    
    if not os.path.exists(workDir):
        print('No existe' + workDir)
    else:
        print(str(i)+'/'+str(len(all_dir))+'---------------------------------------------------------------------------------------------')
        print(workDir)
    i+=1
    
    # PSF file name
    psfFile = os.path.join(workDir, "nowat.prmtop")
    
    # DCD file name 
    dcdFile = [os.path.join(workDir, "nowat_stride10.dcd")]
    
    u = mda.Universe(psfFile,dcdFile)
    print(len(u.trajectory))
    
    traj_length = len(u.trajectory)
    
    # Segment IDs for regions that will be studied. Amber does not use segID 
    segIDs = ["SYSTEM"]
    
    numWinds = 1 ## only one windonw
    
    # Sampled frames per window 
    numSampledFrames = traj_length
    
    # Network Analysis will make one node per protein residue (in the alpha carbon)
    # For other residues, the user must specify atom(s) that will represent a node.
    customResNodes = {}
    
    # -- Protein residue definition -- ##
    customResNodes["GU1"] = ["CA"]
    customResNodes["CM1"] = ["CA"]
    customResNodes["HD1"] = ["CA"]
    customResNodes["HD2"] = ["CA"]
    customResNodes["CM2"] = ["CA"]
    customResNodes["CY4"] = ["CA"]
    
    # -- NA residue definition -- #
    customResNodes["A5"] = ["N9"] 
    customResNodes["U5"] = ["N1"]
    customResNodes["A"] = ["P"]
    customResNodes["C"] = ["P"]
    customResNodes["G"] = ["P"]
    customResNodes["U"] = ["P"]
    customResNodes["U3"] = ["P"]
    customResNodes["G3"] = ["P"] 
    customResNodes["A3"] = ["P"]
    customResNodes["U3"] = ["P"]
    
    # We also need to know the heavy atoms that compose each node group.
    usrNodeGroups = {}
    
    # -- Protein residue definition -- ##
    customResNodes["GU1"] = ["CA"]
    customResNodes["CM1"] = ["CA"]
    customResNodes["HD1"] = ["CA"]
    customResNodes["HD2"] = ["CA"]
    customResNodes["CM2"] = ["CA"]
    customResNodes["CY4"] = ["CA"]
    
    # -- NA residue definition -- #
    customResNodes["A5"] = ["N9"] 
    customResNodes["U5"] = ["N1"]
    customResNodes["A"] = ["P"]
    customResNodes["C"] = ["P"]
    customResNodes["G"] = ["P"]
    customResNodes["U"] = ["P"]
    customResNodes["U3"] = ["P"]
    customResNodes["G3"] = ["P"] 
    customResNodes["A3"] = ["P"]
    customResNodes["U3"] = ["P"]
    
    # We also need to know the heavy atoms that compose each node group.
    usrNodeGroups = {}
    
    # usrNodeGroups["G5"] = {}
    usrNodeGroups["GU1"] = {}
    usrNodeGroups["CM1"] = {}
    usrNodeGroups["HD1"] = {}
    usrNodeGroups["HD2"] = {}
    usrNodeGroups["CM2"] = {}
    usrNodeGroups["CY4"] = {}
    usrNodeGroups["A5"] = {}
    usrNodeGroups["U5"] = {}
    usrNodeGroups["A"] = {}
    usrNodeGroups["C"] = {}
    usrNodeGroups["G"] = {}
    usrNodeGroups["U"] = {}
    usrNodeGroups["U3"] = {}
    usrNodeGroups["G3"] = {}
    usrNodeGroups["A3"] = {}
    usrNodeGroups["C3"] = {}
    
    # usrNodeGroups["G5"]["N9"] = set("O5' C5' C4' O4' C1' N9 C3' C2' O2' O3'".split())
    usrNodeGroups["GU1"]["CA"] = set("C O CA N CB".split())
    usrNodeGroups["CM1"]["CA"] = set("C O CA N CB".split())
    usrNodeGroups["HD1"]["CA"] = set("C O CA N CB".split())
    usrNodeGroups["HD2"]["CA"] = set("C O CA N CB".split())
    usrNodeGroups["CM2"]["CA"] = set("C O CA N CB".split())
    usrNodeGroups["CY4"]["CA"] = set("C O CA N CB".split())
    usrNodeGroups["A5"]["N9"] = set("O5' C5' C4' O4' C1' N9 C3' C2' O2' O3'".split())
    usrNodeGroups["U5"]["N1"] = set("O5' C5' C4' O4' C1' N1 C3' C2' O2' O3'".split())
    usrNodeGroups["A"]["P"] = set("P OP1 OP2 O5' C5' C4' O4' C1' C3' C2' O2' O3'".split())
    usrNodeGroups["C"]["P"] = set("P OP1 OP2 O5' C5' C4' O4' C1' C3' C2' O2' O3'".split())
    usrNodeGroups["G"]["P"] = set("P OP1 OP2 O5' C5' C4' O4' C1' C3' C2' O2' O3'".split())
    usrNodeGroups["U"]["P"] = set("P OP1 OP2 O5' C5' C4' O4' C1' C3' C2' O2' O3'".split())
    usrNodeGroups["U3"]["P"] = set("P OP1 OP2 O5' C5' C4' O4' C1' C3' C2' O2' O3'".split())
    usrNodeGroups["G3"]["P"] = set("P OP1 OP2 O5' C5' C4' O4' C1' C3' C2' O2' O3'".split())
    usrNodeGroups["A3"]["P"] = set("P OP1 OP2 O5' C5' C4' O4' C1' C3' C2' O2' O3'".split())
    usrNodeGroups["C3"]["P"] = set("P OP1 OP2 O5' C5' C4' O4' C1' C3' C2' O2' O3'".split())
    
    
    #############################################
    ### Configuration to filter
    
    # Cutoff for contact map (In Angstroms)
    cutoffDist = 4.5
    
    # Minimum contact persistance (In ratio of total trajectory frames)
    contactPersistence = 0.75
    
    #################################
    ### Load info to object
    
    dnap.setNumWinds(numWinds)
    dnap.setNumSampledFrames(numSampledFrames)
    dnap.setCutoffDist(cutoffDist)
    dnap.setContactPersistence(contactPersistence)
    dnap.setSegIDs(segIDs)
    dnap.setNodeGroups(usrNodeGroups)
    
    dnap.loadSystem(psfFile,dcdFile)
    dnap.getU().trajectory
    
    dnap.checkSystem()
   
    dnap.prepareNetwork()
    
    # If your system is too large, you can turn off the "in memory" option, at a cost for performance.
    dnap.alignTraj(inMemory=True)

    # To speed-up the contact matrix calculation, a larger stride can be selected, at a cost for precision.
    # dnap.findContacts(stride=1)
    dnap.findContacts(stride=10) ## to check whether stride impacts GC
    print(workDir)
    print("Contacts finished, will calculate GC")
    dnap.filterContacts(notSameRes=True, notConsecutiveRes=False, removeIsolatedNodes=True)
    
    dnap.calcCor(ncores=36)

    dnap.calcCartesian(backend="serial")
    
    # Create a variable to save the GC martix
    n1_R1 = ()
    n1_R1 = dnap.corrMatAll
    
    # GC matrix (first variable is the number of windows)
    print(n1_R1.shape)
    
    # Save the GC matrix as a np array
    nameFileGC = "GC_npArray_filtered.npy"
    np.save(workDir+nameFileGC, n1_R1)

    print ('Contacts & Graph ######################################################################')

    dnap.calcGraphInfo()
    # Basic information of the network as interpreted as a graph.
    print(dnap.nxGraphs[0])

    # Both density and transitivity are scaled from 0 to 1
    for win in range(dnap.numWinds):
        print("----- Window {} -----".format(win))
        print("Density:", round( nx.density(dnap.nxGraphs[win]), 4) )
        print("Transitivity:", round( nx.transitivity(dnap.nxGraphs[win]), 4) )
        print()
    from operator import itemgetter

    # We can check the nodes that have the most connections in each window.
    for win in range(dnap.numWinds):
        print("----- Window {} -----".format(win))
        
        sorted_degree = sorted(dnap.getDegreeDict(win).items(), key=itemgetter(1), reverse=True)
        
        print("Top 10 nodes by degree: [node --> degree : selection]")
        for n,d in sorted_degree[:10]:
            print("{0:>4} --> {1:>2} : {2}".format(n, d, getSelFromNode(n, dnap.nodesAtmSel)))
        
        print()

    dnap.calcOptPaths (ncores=36)
    
    dnap.calcBetween (ncores=36)
    
    from itertools import islice

    # Pairs of nodes with highest Betweeness values, compared to their correlation values (in Window 0)
    for k,v in islice(dnap.btws[0].items(),10):
        print("Nodes {} have betweennes {} and correlation {}.".format(k, round(v,3), round(dnap.corrMatAll[0, k[0], k[1]], 3) ) )

    dnap.calcEigenCentral()
    dnap.calcCommunities()

    # Sort communities based on number of nodes
    for comIndx in dnap.nodesComm[0]["commOrderSize"]:
        print("Modularity Class {0:>2}: {1:>3} nodes.".format(comIndx, len(dnap.nodesComm[0]["commNodes"][comIndx])))

    # Sort communities based on the node with highest eigenvector centrality
    for comIndx in dnap.nodesComm[0]["commOrderEigenCentr"]:
        print("Modularity Class {0} ({1} nodes) Sorted by Eigenvector Centrality:".format(
                                                                        comIndx, 
                                                                    len(dnap.nodesComm[0]["commNodes"][comIndx])))
        for node in dnap.nodesComm[0]["commNodes"][comIndx][:5]:
            print("Name: {0:>4} | Degree: {1:>2} | Eigenvector Centrality: {2}".format(
                node, dnap.nxGraphs[win].nodes[node]['degree'], dnap.nxGraphs[win].nodes[node]['eigenvector']))
        print()
    
    pathToData = workDir+"Results_gc_filtered"
    
    if not os.path.isdir(pathToData):
        os.mkdir(pathToData)
        
    fileNameRoot = "Data_"
    
    fullPathRoot = os.path.join(pathToData, fileNameRoot)
    
    dnap.saveData(fullPathRoot)

    # This function will save a reduced DCD trajectory with the heavy atoms used for network analysis
    # A smaller trajectory can be created by choosing a "stride" that sub-samples the original trajectory.
    # This function will also produce a PDB file so that information on atoms and residues can be loaded to
    #    visualization software such as VMD.
    
    dcdstride = 10
    
    print("We will save {} heavy atoms and {} frames.".format(dnap.workU.atoms.n_atoms, len(dnap.workU.trajectory[::dcdstride]) ))

    dnap.saveReducedTraj(fullPathRoot, stride = 10)


1/1---------------------------------------------------------------------------------------------
/home/usuario/ns3/qmmm/paper/md/atp/analisis/
1250
Residue verification:

---> SegID  SYSTEM : 21 unique residue types:
{'CYS', 'ILE', 'ASP', 'GLU', 'THR', 'TYR', 'LYS', 'PHE', 'ALA', 'GLN', 'MET', 'TRP', 'ASN', 'GLY', 'VAL', 'HIE', 'ARG', 'HID', 'LEU', 'SER', 'PRO'}

---> 21 total selected residue types:
{'ASN', 'CYS', 'ILE', 'GLY', 'ASP', 'GLU', 'THR', 'VAL', 'HIE', 'TYR', 'ARG', 'LYS', 'HID', 'LEU', 'PHE', 'ALA', 'GLN', 'MET', 'TRP', 'SER', 'PRO'}

---> 0 not-selected residue types in other segments:
set()

---> 21 total residue types:
{'ASN', 'CYS', 'ILE', 'GLY', 'ASP', 'GLU', 'THR', 'VAL', 'HIE', 'TYR', 'ARG', 'LYS', 'HID', 'LEU', 'PHE', 'ALA', 'GLN', 'MET', 'TRP', 'SER', 'PRO'}

---> 439 total residues were selected for network analysis.

Segments verification:

---> 1 selected segments:
['SYSTEM']

---> 0 not-selected segments:
[]



Nodes:   0%|          | 0/439 [00:00<?, ?it/s]

Window:   0%|          | 0/1 [00:00<?, ?it/s]

We found 0 nodes with no contacts.
We found 2213 contacting pairs out of 96141 total pairs of nodes.
(That's 2.3%, by the way)
/home/usuario/ns3/qmmm/paper/md/atp/analisis/
Contacts finished, will calculate GC
Filtering contacts in each window.


Window:   0%|          | 0/1 [00:00<?, ?it/s]

...

Window:   0%|          | 0/1 [00:00<?, ?it/s]

Window:   0%|          | 0/1 [00:00<?, ?it/s]

Nodes (185, 423) have betweennes 0.051 and correlation 0.147.
Nodes (209, 425) have betweennes 0.039 and correlation 0.362.
Nodes (107, 117) have betweennes 0.029 and correlation 0.501.
Nodes (111, 419) have betweennes 0.029 and correlation 0.395.
Nodes (208, 210) have betweennes 0.027 and correlation 0.478.
Nodes (419, 423) have betweennes 0.026 and correlation 0.881.
Nodes (108, 230) have betweennes 0.025 and correlation 0.296.
Nodes (85, 87) have betweennes 0.025 and correlation 0.712.
Nodes (208, 426) have betweennes 0.023 and correlation 0.179.
Nodes (111, 116) have betweennes 0.022 and correlation 0.848.
Modularity Class  6:  81 nodes.
Modularity Class  0:  60 nodes.
Modularity Class  5:  59 nodes.
Modularity Class  3:  58 nodes.
Modularity Class  7:  55 nodes.
Modularity Class  1:  43 nodes.
Modularity Class  2:  32 nodes.
Modularity Class  8:  27 nodes.
Modularity Class  4:  24 nodes.
Modularity Class 2 (32 nodes) Sorted by Eigenvector Centrality:
Name:  353 | Degree: 17 | Eige

Frames:   0%|          | 0/125 [00:00<?, ?it/s]

/home/usuario/Programas/anaconda3/lib/python3.12/site-packages/MDAnalysis/coordinates/DCD.py:463: UserWarning: No dimensions set for current frame, zeroed unitcell will be written
  warnings.warn(wmsg)
/home/usuario/Programas/anaconda3/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:777: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn("Unit cell dimensions not found. "
/home/usuario/Programas/anaconda3/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1154: UserWarning: Found no information for attr: 'altLocs' Using default value of ' '
  warnings.warn("Found no information for attr: '{}'"
/home/usuario/Programas/anaconda3/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1154: UserWarning: Found no information for attr: 'icodes' Using default value of ' '
  warnings.warn("Found no information for attr: '{}'"
/home/usuario/Programas/anaconda3/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1154: U